# Expected stat curve

Load a saved policy task result, choose a statistic and one parameter dimension, then plot the final-step expected statistic after averaging over the remaining sampled parameters.


In [1]:
import sys
from dataclasses import dataclass
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from IPython.display import HTML, clear_output, display

project_root = Path.cwd()
if not (project_root / "tasks").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

try:
    import scienceplots  # noqa: F401
except ModuleNotFoundError:
    pass
else:
    plt.style.use("science")

from tasks.policies import load_policy_results

In [2]:
@dataclass(frozen=True)
class CurveConfig:
    result_store: str | Path | None = None
    stat: str | None = None
    dimension: str | None = None
    rolling_window: int = 501
    low_cardinality_max_unique: int = 20
    scatter_limit: int = 8000
    point_size: float = 8.0
    point_alpha: float = 0.08


@dataclass(frozen=True)
class CurveContext:
    project_root: Path
    result_store_path: Path
    ds: xr.Dataset
    proportion_dim: str
    theta_labels: tuple[str, ...]
    observable_labels: tuple[str, ...]
    dimension_labels: tuple[str, ...]


@dataclass(frozen=True)
class CurveSamples:
    stat: str
    dimension: str
    x: np.ndarray
    y: np.ndarray


@dataclass(frozen=True)
class CurveData:
    stat: str
    dimension: str
    mode: str
    x: np.ndarray
    y: np.ndarray
    curve_x: np.ndarray
    curve_y: np.ndarray
    curve_sem: np.ndarray
    sample_count: int
    unique_count: int
    window: int | None
    run_name: str

In [3]:
def result_store_candidates(root):
    return sorted(
        root.glob("artifacts/**/*.zarr"), key=lambda p: p.stat().st_mtime, reverse=True
    )


def resolve_result_store(root, value):
    if value is not None:
        path = Path(value).expanduser()
        return path if path.is_absolute() else root / path
    candidates = result_store_candidates(root)
    if not candidates:
        raise FileNotFoundError("No .zarr result stores found under artifacts/")
    return candidates[0]


def unique_preserving_order(values):
    return tuple(dict.fromkeys(str(value) for value in values))


def load_context(config, root):
    ds = load_policy_results(resolve_result_store(root, config.result_store))
    proportion_dims = tuple(dim for dim in ds["x"].dims if dim.endswith("_proportion"))
    if len(proportion_dims) != 1:
        raise ValueError(
            f"Expected one policy-proportion dimension, found {proportion_dims}"
        )

    proportion_dim = proportion_dims[0]
    theta_labels = tuple(str(value) for value in ds.coords["theta_parameter"].values)
    observable_labels = tuple(str(value) for value in ds.coords["observable"].values)
    dimension_labels = unique_preserving_order((*theta_labels, proportion_dim))

    return CurveContext(
        project_root=root,
        result_store_path=Path(ds.attrs["result_store_path"]),
        ds=ds,
        proportion_dim=proportion_dim,
        theta_labels=theta_labels,
        observable_labels=observable_labels,
        dimension_labels=dimension_labels,
    )

In [ ]:
def selected_stat_values(ctx, stat):
    if stat not in ctx.observable_labels:
        raise ValueError(f"Unknown stat {stat!r}. Available: {ctx.observable_labels}")
    return ctx.ds["x"].sel(observable=stat).isel(step=-1, drop=True)


def selected_dimension_values(ctx, dimension, template):
    if dimension in ctx.theta_labels:
        return ctx.ds["theta"].sel(theta_parameter=dimension, drop=True)
    if dimension == ctx.proportion_dim:
        _, values = xr.broadcast(template, ctx.ds.coords[ctx.proportion_dim])
        return values
    raise ValueError(
        f"Unknown dimension {dimension!r}. Available: {ctx.dimension_labels}"
    )


def flatten_curve_samples(ctx, stat, dimension):
    y_data = selected_stat_values(ctx, stat)
    x_data = selected_dimension_values(ctx, dimension, y_data)
    x_data, y_data = xr.align(x_data, y_data, join="exact")

    x = np.asarray(x_data.values, dtype=float).ravel()
    y = np.asarray(y_data.values, dtype=float).ravel()
    mask = np.isfinite(x) & np.isfinite(y)
    if not mask.any():
        raise ValueError(f"No finite final-step values for {stat!r} by {dimension!r}.")
    return CurveSamples(stat=stat, dimension=dimension, x=x[mask], y=y[mask])


def grouped_mean_curve(x, y):
    unique_x = np.unique(x)
    curve_y = np.empty(unique_x.shape, dtype=float)
    curve_sem = np.zeros(unique_x.shape, dtype=float)

    for index, value in enumerate(unique_x):
        values = y[x == value]
        curve_y[index] = float(np.mean(values))
        if values.size > 1:
            curve_sem[index] = float(np.std(values, ddof=1) / np.sqrt(values.size))

    return unique_x, curve_y, curve_sem


def normalized_window(value, sample_count):
    window = max(1, min(int(value), int(sample_count)))
    return window


def rolling_mean_curve(x, y, window):
    order = np.argsort(x, kind="mergesort")
    sorted_x = x[order]
    sorted_y = y[order]
    n = sorted_y.size
    window = normalized_window(window, n)
    half_window = window // 2

    starts = np.arange(n, dtype=np.int64) - half_window
    starts = np.clip(starts, 0, max(0, n - window))
    stops = np.minimum(starts + window, n)
    counts = stops - starts

    sums = np.concatenate([[0.0], np.cumsum(sorted_y)])
    sums_sq = np.concatenate([[0.0], np.cumsum(sorted_y * sorted_y)])
    window_sums = sums[stops] - sums[starts]
    window_sums_sq = sums_sq[stops] - sums_sq[starts]

    means = window_sums / counts
    sem = np.zeros(n, dtype=float)
    variance_mask = counts > 1
    variance = np.zeros(n, dtype=float)
    variance[variance_mask] = (
        window_sums_sq[variance_mask]
        - (
            window_sums[variance_mask]
            * window_sums[variance_mask]
            / counts[variance_mask]
        )
    ) / (counts[variance_mask] - 1)
    sem[variance_mask] = np.sqrt(
        np.maximum(variance[variance_mask], 0.0) / counts[variance_mask]
    )

    return sorted_x, means, sem, window


def build_curve_data(
    ctx, stat, dimension, *, rolling_window, low_cardinality_max_unique
):
    samples = flatten_curve_samples(ctx, stat, dimension)
    unique_count = int(np.unique(samples.x).size)
    run_name = ctx.result_store_path.stem

    if unique_count <= int(low_cardinality_max_unique):
        curve_x, curve_y, curve_sem = grouped_mean_curve(samples.x, samples.y)
        return CurveData(
            stat=stat,
            dimension=dimension,
            mode="grouped mean",
            x=samples.x,
            y=samples.y,
            curve_x=curve_x,
            curve_y=curve_y,
            curve_sem=curve_sem,
            sample_count=samples.y.size,
            unique_count=unique_count,
            window=None,
            run_name=run_name,
        )

    curve_x, curve_y, curve_sem, window = rolling_mean_curve(
        samples.x, samples.y, rolling_window
    )
    return CurveData(
        stat=stat,
        dimension=dimension,
        mode="rolling mean",
        x=samples.x,
        y=samples.y,
        curve_x=curve_x,
        curve_y=curve_y,
        curve_sem=curve_sem,
        sample_count=samples.y.size,
        unique_count=unique_count,
        window=window,
        run_name=run_name,
    )

In [ ]:
def downsample_indices(count, limit):
    if count <= limit:
        return np.arange(count, dtype=np.int64)
    return np.unique(np.linspace(0, count - 1, int(limit), dtype=np.int64))


def curve_ylabel(stat, dimension):
    return f"E[{stat} | {dimension}]"


def plot_expected_curve(
    data,
    config,
    *,
    fig=None,
    figsize=(8, 4.5),
    color="tab:blue",
    show_sem=True,
):
    if fig is None:
        with plt.ioff():
            fig = plt.figure(figsize=figsize)
    else:
        fig.clear()
        fig.set_size_inches(*figsize, forward=True)

    ax = fig.add_subplot(1, 1, 1)
    if data.mode == "rolling mean":
        order = np.argsort(data.x, kind="mergesort")
        point_indices = order[
            downsample_indices(data.sample_count, config.scatter_limit)
        ]
        ax.scatter(
            data.x[point_indices],
            data.y[point_indices],
            s=config.point_size,
            alpha=config.point_alpha,
            color="0.25",
            linewidths=0,
            rasterized=True,
            label="final samples",
        )

    ax.plot(data.curve_x, data.curve_y, color=color, linewidth=2.0, label=data.mode)
    if show_sem and np.any(np.isfinite(data.curve_sem)):
        ax.fill_between(
            data.curve_x,
            data.curve_y - data.curve_sem,
            data.curve_y + data.curve_sem,
            color=color,
            alpha=0.18,
            linewidth=0,
            label="standard error",
        )

    mode_detail = (
        data.mode if data.window is None else f"{data.mode}, window={data.window}"
    )
    ax.set_title(
        f"{data.stat}\nby {data.dimension} from {data.run_name} "
        f"({data.sample_count:,} samples, {data.unique_count:,} unique x, {mode_detail})"
    )
    ax.set_xlabel(data.dimension)
    ax.set_ylabel(curve_ylabel(data.stat, data.dimension))
    ax.grid(alpha=0.25)
    ax.legend(loc="best")
    fig.tight_layout()
    return fig, ax

In [ ]:
def relative_label(root, path):
    try:
        return str(path.relative_to(root))
    except ValueError:
        return str(path)


def result_store_options(root):
    candidates = result_store_candidates(root)
    if not candidates:
        raise FileNotFoundError("No .zarr result stores found under artifacts/")
    return [
        (relative_label(root, path), relative_label(root, path)) for path in candidates
    ]


def set_options(control, options, preferred=None):
    values = [item[1] if isinstance(item, tuple) else item for item in options]
    control.options = options
    if preferred in values:
        control.value = preferred
    elif control.value not in values:
        control.value = values[0]


def labeled(title, widget):
    return widgets.VBox([widgets.HTML(f"<b>{title}</b>"), widget])


def make_curve_widget(config, root):
    store = widgets.Dropdown(
        options=result_store_options(root),
        description="",
        layout=widgets.Layout(width="520px"),
    )
    if config.result_store is not None:
        resolved = resolve_result_store(root, config.result_store)
        relative = relative_label(root, resolved)
        if relative not in [value for _, value in store.options]:
            store.options = ((relative, relative), *tuple(store.options))
        store.value = relative

    stat = widgets.Select(rows=6, layout=widgets.Layout(width="320px", height="150px"))
    dimension = widgets.Select(
        rows=8, layout=widgets.Layout(width="260px", height="190px")
    )
    rolling_window = widgets.BoundedIntText(
        value=int(config.rolling_window),
        min=1,
        max=100000,
        step=2,
        layout=widgets.Layout(width="110px"),
    )
    plot_area = widgets.Output()
    status = widgets.HTML()

    state = {"ctx": None, "fig": None, "updating": False}

    def load_selected_context():
        selected_config = CurveConfig(result_store=store.value)
        ctx = load_context(selected_config, root)
        state["ctx"] = ctx
        status.value = f"Loaded <code>{relative_label(ctx.project_root, ctx.result_store_path)}</code>"
        return ctx

    def refresh_options(*_):
        if state["updating"]:
            return
        state["updating"] = True
        try:
            ctx = load_selected_context()
            set_options(stat, list(ctx.observable_labels), preferred=config.stat)
            set_options(
                dimension, list(ctx.dimension_labels), preferred=config.dimension
            )
        finally:
            state["updating"] = False
        draw()

    def draw(*_):
        if state["updating"] or state["ctx"] is None:
            return
        ctx = state["ctx"]
        try:
            data = build_curve_data(
                ctx,
                stat.value,
                dimension.value,
                rolling_window=rolling_window.value,
                low_cardinality_max_unique=config.low_cardinality_max_unique,
            )
            state["fig"], _ = plot_expected_curve(data, config, fig=state["fig"])
            with plot_area:
                clear_output(wait=True)
                display(state["fig"])
        except Exception as exc:
            with plot_area:
                clear_output(wait=True)
                display(HTML(f"<pre>{type(exc).__name__}: {exc}</pre>"))

    store.observe(refresh_options, names="value")
    stat.observe(draw, names="value")
    dimension.observe(draw, names="value")
    rolling_window.observe(draw, names="value")

    refresh_options()

    controls = widgets.HBox(
        [
            labeled("Result store", store),
            labeled("Stat", stat),
            labeled("Dimension", dimension),
            labeled("Rolling window", rolling_window),
        ],
        layout=widgets.Layout(
            gap="18px", align_items="flex-start", flex_flow="row wrap"
        ),
    )
    return widgets.VBox([controls, status, plot_area])

In [ ]:
config = CurveConfig(
    result_store=None,
    stat=None,
    dimension=None,
    rolling_window=501,
    low_cardinality_max_unique=20,
    scatter_limit=8000,
)

display(make_curve_widget(config, project_root))